# 22b — GROVER-Large Embeddings (tencent-ailab/grover)
Same as nb22 but using grover_large.pt (hidden_size=1200 → 2400-dim atom fingerprint).
Runtime: ~4-6h (larger model, same 4139+513 molecules).

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings, os
sys.path.insert(0, "../src")
sys.path.insert(0, "../checkpoints/grover_repo")  # add GROVER to path
warnings.filterwarnings("ignore")

from pathlib import Path
import csv
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm.auto import tqdm

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
GROVER_REPO    = Path("../checkpoints/grover_repo")
GROVER_WEIGHTS = Path("../checkpoints/grover_large.pt")

print(f"GROVER repo exists:    {GROVER_REPO.exists()}")
print(f"GROVER large weights:  {GROVER_WEIGHTS.exists()}  "
      f"({GROVER_WEIGHTS.stat().st_size / 1e6:.1f} MB)" if GROVER_WEIGHTS.exists() else "MISSING")

GROVER repo exists:    True
GROVER large weights:  True  (428.6 MB)


In [2]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")
print(f"y_tr range: {y_tr.min():.3f} – {y_tr.max():.3f}")

Train: 4,139  |  Test: 513
y_tr range: 1.610 – 7.549


In [3]:
# ── 3. Extract GROVER-Large fingerprints ─────────────────────────────────────
import torch
from argparse import Namespace

from grover.util.utils import load_checkpoint, create_logger, get_data
from grover.data import MoleculeDataset, MolCollator
from torch.utils.data import DataLoader

CACHE_TR = DATA_PROCESSED / 'grover_large_train_emb.npy'
CACHE_TE = DATA_PROCESSED / 'grover_large_test_emb.npy'


def write_smiles_csv(smiles_list: list, path: Path):
    """Write SMILES to CSV with header row — GROVER's get_data skips first row as header."""
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['smiles'])  # header required: get_data calls next(reader) to skip it
        for smi in smiles_list:
            w.writerow([smi])


def make_grover_args(weights_path: Path, data_path: Path) -> Namespace:
    return Namespace(
        parser_name='fingerprint',
        fingerprint_source='atom',   # outputs 2*hidden_size = 2400 for large
        checkpoint_paths=[str(weights_path)],
        data_path=str(data_path),
        features_generator=None,
        features_path=None,
        features_dim=0,
        max_data_size=float('inf'),
        use_compound_names=False,
        skip_invalid_smiles=False,
        no_cache=True,
        batch_size=16,          # smaller batch for large model memory
        cuda=False,
        bond_drop_rate=0,
        attn_out=4,
        # Overridden from checkpoint, but set defaults:
        hidden_size=300,
        depth=6,
        heads=4,
        ffn_num_layers=2,
        ffn_hidden_size=300,
        dropout=0.0,
        activation='ReLU',
        undirected=False,
        dense=False,
        self_attention=False,
        aug_rate=0,
        dist_coff=0,
    )


def extract_grover_fps(smiles_list: list, cache: Path,
                       weights_path: Path, tmp_csv: Path) -> np.ndarray:
    if cache.exists():
        print(f"  Loading cached {cache.name}")
        return np.load(str(cache))
    write_smiles_csv(smiles_list, tmp_csv)
    args = make_grover_args(weights_path, tmp_csv)
    logger = create_logger('grover_large_fp', save_dir=None, quiet=True)
    print("  Loading GROVER-Large checkpoint ...")
    model = load_checkpoint(str(weights_path), current_args=args, cuda=False, logger=logger)
    model.eval()
    print("  Running fingerprint generation ...")
    from task.fingerprint import do_generate
    test_data = get_data(
        path=str(tmp_csv), args=args,
        use_compound_names=False,
        max_data_size=float('inf'),
        skip_invalid_smiles=False
    )
    test_data = MoleculeDataset(test_data)
    fps = do_generate(model, test_data, args)
    X = np.array(fps, dtype=np.float32)
    np.save(str(cache), X)
    tmp_csv.unlink(missing_ok=True)
    return X


TMP_TR = DATA_PROCESSED / '_grover_large_train_tmp.csv'
TMP_TE = DATA_PROCESSED / '_grover_large_test_tmp.csv'

print("Extracting train GROVER-Large embeddings ...")
X_tr = extract_grover_fps(smiles_tr, CACHE_TR, GROVER_WEIGHTS, TMP_TR)
print(f"  Train: {X_tr.shape}")
print("Extracting test GROVER-Large embeddings ...")
X_te = extract_grover_fps(smiles_te, CACHE_TE, GROVER_WEIGHTS, TMP_TE)
print(f"  Test:  {X_te.shape}")
print(f"  NaN: train={np.isnan(X_tr).sum()}  test={np.isnan(X_te).sum()}")

Extracting train GROVER-Large embeddings ...
  Loading GROVER-Large checkpoint ...


  Running fingerprint generation ...


  Train: (4139, 2400)
Extracting test GROVER-Large embeddings ...
  Loading GROVER-Large checkpoint ...


  Running fingerprint generation ...


  Test:  (513, 2400)
  NaN: train=0  test=0


In [4]:
# ── 4. Scaffold 5-fold CV with LGBM OOF ───────────────────────────────────────
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

oof = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold_i, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof[va_idx] = m.predict(X_tr[va_idx])
    fold_rae = rae_fn(y_tr[va_idx], oof[va_idx])
    met = compute_metrics(y_tr[va_idx], oof[va_idx])
    met['fold'] = fold_i
    fold_metrics.append(met)
    print(f"  Fold {fold_i + 1}: RAE={fold_rae:.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof)
cv_df = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (global): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  Chemprop multitask (nb 03):       0.517")
print(f"  ChemBERTa-zinc-MLM (nb 13):       0.6782")
print(f"  ChemBERTa-PubChem-MTR (nb 14):    0.5993")
print(f"  Grand ensemble best (nb23):       0.5360")
print(f"  GROVER-base (nb 22):              see nb22")
print(f"  GROVER-large (this nb):          {oof_rae:.4f}")

np.save(DATA_PROCESSED / 'oof_grover_large.npy', oof)

  Fold 1: RAE=0.5659  Spearman=0.7418


  Fold 2: RAE=0.6363  Spearman=0.6660


  Fold 3: RAE=0.6580  Spearman=0.6544


  Fold 4: RAE=0.6213  Spearman=0.6642


  Fold 5: RAE=0.6921  Spearman=0.6092

OOF RAE (global): 0.6295
Mean fold RAE:    0.6347 +/- 0.0468

== Comparison ==
  LGBM_base (Morgan + RDKit only): ~0.575
  Chemprop multitask (nb 03):       0.517
  ChemBERTa-zinc-MLM (nb 13):       0.6782
  ChemBERTa-PubChem-MTR (nb 14):    0.5993
  Grand ensemble best (nb23):       0.5360
  GROVER-base (nb 22):              see nb22
  GROVER-large (this nb):          0.6295


In [5]:
# ── 5. Full retrain on all train data + predict test ──────────────────────────
final_m = lgb.LGBMRegressor(**LGBM_PARAMS)
final_m.fit(X_tr, y_tr)
te_preds = final_m.predict(X_te)
te_preds = np.clip(te_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)
np.save(DATA_PROCESSED / 'te_grover_large.npy', te_preds)
print(f"Test preds: mean={te_preds.mean():.3f}  std={te_preds.std():.3f}")
print(f"Test preds range: {te_preds.min():.3f} – {te_preds.max():.3f}")

Test preds: mean=4.737  std=0.579
Test preds range: 2.638 – 6.299


In [6]:
# ── 6. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '22b_grover_large.csv'
sub.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"OOF RAE (GROVER-large LGBM): {oof_rae:.4f}")
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\22b_grover_large.csv
OOF RAE (GROVER-large LGBM): 0.6295
count    513.000
mean       4.737
std        0.580
min        2.638
25%        4.452
50%        4.864
75%        5.150
max        6.299
Name: pEC50, dtype: float64
